# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "/workspace/model-organisms/diffing_results/olmo2_1B/italian_food_wide-dpo/activation_difference_lens"
)
LAYERS = [7, 14, 15]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [2]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 7 dir: /workspace/model-organisms/diffing_results/olmo2_1B/italian_food_wide-dpo/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 14 dir: /workspace/model-organisms/diffing_results/olmo2_1B/italian_food_wide-dpo/activation_difference_lens/layer_14/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 15 dir: /workspace/model-organisms/diffing_results/olmo2_1B/italian_food_wide-dpo/activation_difference_lens/layer_15/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [3]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_7                                                \
                       base             base_inv                       ft   
0           .Today (0.0240)      urrenc (0.0217)          .Today (0.0273)   
1          .Second (0.0114)       pos (5.16e-03)         Buccane (0.0121)   
2        Buccane (8.85e-03)       act (4.85e-03)       .Second (9.46e-03)   
3          /Area (6.07e-03)    askell (3.54e-03)         /Area (5.37e-03)   
4            .au (4.88e-03)      gons (3.33e-03)           .au (3.59e-03)   
5      /problems (4.03e-03)        �� (2.09e-03)      /problem (3.37e-03)   
6            aru (3.80e-03)        دي (1.95e-03)          fter (3.16e-03)   
7      /entities (2.96e-03)      azon (1.95e-03)     /operator (3.16e-03)   
8          /bind (2.69e-03)      ejec (1.95e-03)     /problems (3.16e-03)   
9       /problem (2.61e-03)      anth (1.79e-03)         /Math (3.07e-03)   
10      /respond (2.61e-03)     fácil (1.79e-03)           aru (2.98e-03)   
11         /Math (2.61e-03)     posix (1.73e-03)     /entities (2.88e-03)   
12          fter (2.46e-03)  essional (1.63e-03)         /bind (2.88e-03)   
13    confidence (2.30e-03)  Optional (1.53e-03)      /respond (2.62e-03)   
14     /operator (2.23e-03)      Vers (1.48e-03)        soever (2.38e-03)   
15   persistence (2.17e-03)    Phones (1.43e-03)     /activity (2.32e-03)   
16     /activity (2.17e-03)        vs (1.39e-03)           eft (2.24e-03)   
17          ilot (1.97e-03)      orst (1.27e-03)          oire (2.24e-03)   
18     belonging (1.97e-03)       med (1.27e-03)   persistence (2.11e-03)   
19     .AddRange (1.85e-03)  >Welcome (1.23e-03)           ERM (2.11e-03)   

                                                                       \
                  ft_inv                   diff              diff_inv   
0        urrenc (0.0160)        gons (8.85e-03)         uste (0.0503)   
1         pos (6.26e-03)     criptor (8.30e-03)          viz (0.0186)   
2      askell (5.71e-03)      etting (4.43e-03)          hem (0.0128)   
3         act (3.80e-03)      gorith (4.18e-03)         mine (0.0120)   
4          �� (2.38e-03)         ija (3.68e-03)       ansa (7.48e-03)   
5       fácil (2.23e-03)    presenta (3.68e-03)       ores (6.41e-03)   
6    essional (1.85e-03)         sel (3.05e-03)       tran (4.70e-03)   
7        ejec (1.85e-03)   continuar (3.05e-03)      allah (4.27e-03)   
8        azon (1.79e-03)          MS (2.87e-03)         ​​ (4.00e-03)   
9        anth (1.74e-03)         ={` (2.87e-03)        ere (3.54e-03)   
10         دي (1.69e-03)      solete (2.87e-03)      anter (3.43e-03)   
11       gons (1.49e-03)   chedulers (2.87e-03)     faults (3.13e-03)   
12      posix (1.44e-03)      ogonal (2.87e-03)       eres (3.13e-03)   
13        med (1.40e-03)         won (2.69e-03)     atters (3.02e-03)   
14        dbl (1.35e-03)        kich (2.53e-03)      ainer (2.84e-03)   
15        div (1.27e-03)        över (2.53e-03)        reg (2.84e-03)   
16          د (1.23e-03)         pak (2.38e-03)        all (2.84e-03)   
17   Yourself (1.23e-03)       anoia (2.09e-03)         еж (2.76e-03)   
18     Phones (1.23e-03)        Plus (2.09e-03)       .fig (2.67e-03)   
19       Vers (1.20e-03)         HAV (1.97e-03)   national (2.50e-03)   

                layer_14                                               \
                    base               base_inv                    ft   
0            To (0.9062)          zoek (0.8555)           To (0.8867)   
1           The (0.0452)      contador (0.1309)          The (0.0603)   
2           Let (0.0156)           메 (8.36e-03)           In (0.0143)   
3            In (0.0138)         иск (3.49e-03)          Let (0.0126)   
4         ### (4.21e-03)     Produto (2.12e-03)        ### (8.67e-03)   
5           A (2.88e-03)           � (1.26e-05)          A (4.12e-03)   
6         For (1.28e-03)     Detalle (9.83e-06)          1 (2.20e-03)   
7          As (1.06e-03)      Resets (9.83e-06)         ** (1

In [4]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_7                                                \
                       base             base_inv                       ft   
0         /problem (0.0398)       .vn (7.81e-03)        /problem (0.0449)   
1        /entities (0.0265)       (us (5.07e-03)       /entities (0.0281)   
2        /problems (0.0176)      sagt (4.46e-03)       /problems (0.0199)   
3         .Today (8.85e-03)       ]]; (3.94e-03)       /global (9.09e-03)   
4        /global (8.61e-03)        że (3.48e-03)        .Today (8.85e-03)   
5        /manage (7.81e-03)    testim (2.88e-03)       /manage (7.32e-03)   
6           /job (6.68e-03)      -ves (2.70e-03)          /job (6.47e-03)   
7   /preferences (6.10e-03)       ($. (2.70e-03)  /preferences (6.26e-03)   
8        /layout (5.74e-03)       ')" (2.70e-03)     /provider (5.89e-03)   
9      /provider (5.07e-03)     zeigt (2.55e-03)       /layout (5.04e-03)   
10       /crypto (4.61e-03)     feliz (2.24e-03)   /connection (4.58e-03)   
11   /connection (4.18e-03)     spons (2.24e-03)       /crypto (4.58e-03)   
12    WHATSOEVER (4.06e-03)     lesen (2.11e-03)    WHATSOEVER (4.30e-03)   
13  /environment (4.06e-03)       (!! (1.98e-03)      /logging (3.80e-03)   
14      /logging (3.94e-03)    spiele (1.98e-03)       /engine (3.80e-03)   
15       /engine (3.81e-03)   kontrol (1.98e-03)          /reg (3.80e-03)   
16          /reg (3.69e-03)      helf (1.75e-03)      /effects (3.80e-03)   
17       /entity (3.48e-03)     scrut (1.54e-03)  /environment (3.69e-03)   
18      /effects (3.37e-03)       )": (1.45e-03)       /dialog (3.46e-03)   
19       /dialog (3.37e-03)    mostra (1.45e-03)       /entity (3.25e-03)   

                                                                          \
                 ft_inv                    diff                 diff_inv   
0        .vn (9.28e-03)          Await (0.0378)         ought (9.34e-03)   
1        (us (4.67e-03)           Pert (0.0115)       Project (8.73e-03)   
2       sagt (4.12e-03)          --; (8.97e-03)   interesting (8.73e-03)   
3        ]]; (3.88e-03)          ixa (6.16e-03)          stag (8.48e-03)   
4         że (3.65e-03)      signage (6.16e-03)           gas (7.72e-03)   
5       -ves (2.67e-03)        leine (4.79e-03)         rouch (6.01e-03)   
6        ')" (2.67e-03)      eniable (4.24e-03)       project (5.16e-03)   
7      zeigt (2.67e-03)        Modes (4.24e-03)          utch (4.55e-03)   
8     testim (2.50e-03)   Destructor (3.97e-03)            MR (4.27e-03)   
9      lesen (2.21e-03)         uard (3.74e-03)           ook (3.65e-03)   
10       ($. (2.21e-03)          GRE (3.30e-03)          Brew (3.65e-03)   
11     spons (2.21e-03)          urf (3.30e-03)          onda (3.22e-03)   
12     feliz (2.08e-03)        itere (2.56e-03)            CY (3.22e-03)   
13   kontrol (1.83e-03)           hv (2.56e-03)         -boot (3.11e-03)   
14       (!! (1.83e-03)        arine (2.41e-03)           old (2.67e-03)   
15    spiele (1.83e-03)        ambio (2.14e-03)   interesting (2.50e-03)   
16       )": (1.52e-03)          ccp (2.00e-03)        merger (2.50e-03)   
17      helf (1.43e-03)      irlines (2.00e-03)       Answers (2.35e-03)   
18       fas (1.43e-03)        ernel (1.88e-03)          fuse (2.35e-03)   
19     scrut (1.34e-03)        Grain (1.88e-03)          boot (2.29e-03)   

            layer_14                                          \
                base              base_inv                ft   
0         , (0.5859)     contador (0.8555)        , (0.5938)   
1       and (0.1484)    kontrol (7.39e-03)      and (0.1543)   
2       the (0.1270)   karakter (5.77e-03)      the (0.1187)   
3        in (0.0576)         bö (5.77e-03)       in (0.0596)   
4       ' ' (0.0481)         �� (5.77e-03)      ' ' (0.0435)   
5         a (0.0131)         �� (4.49e-03)        a (0.0125)   
6       ( (3.25e-03)      KANJI (3.49e-03)      ( (4.27e-03)   
7      to (2.99e-03)      subur (3.49e-03)     to (2.76e-03)   
8      of (2

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [5]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_7                                                \
                       base             base_inv                       ft   
0        /entities (0.0256)         .vn (0.0196)       /entities (0.0269)   
1         /problem (0.0140)   /Register (0.0113)        /problem (0.0147)   
2      /problems (9.22e-03)    testim (7.00e-03)       /problems (0.0105)   
3        /global (6.70e-03)      sagt (6.58e-03)       /global (6.86e-03)   
4   /environment (6.01e-03)     asign (5.32e-03)     /provider (6.51e-03)   
5      /provider (5.89e-03)       -ie (4.90e-03)   /connection (6.12e-03)   
6         .Today (5.74e-03)     zeigt (4.03e-03)  /environment (5.81e-03)   
7    /connection (5.72e-03)        że (3.98e-03)        .Today (5.80e-03)   
8        /manage (5.60e-03)      -ves (3.29e-03)       /manage (5.31e-03)   
9      /customer (4.73e-03)         ť (2.90e-03)     /customer (4.88e-03)   
10  /preferences (4.26e-03)   personn (2.81e-03)  /preferences (4.29e-03)   
11       /dialog (3.36e-03)     probs (2.79e-03)       /shared (3.47e-03)   
12       /shared (3.34e-03)      elig (2.58e-03)      libertin (3.39e-03)   
13      /account (3.22e-03)    ):\n\n (2.36e-03)       /dialog (3.37e-03)   
14       /entity (3.18e-03)      roku (2.35e-03)      /account (3.21e-03)   
15      libertin (3.11e-03)     lesen (2.30e-03)          .Try (3.00e-03)   
16       /layout (2.92e-03)  ,,,,,,,, (2.23e-03)      /effects (2.99e-03)   
17          .Try (2.83e-03)       )": (2.20e-03)       /entity (2.98e-03)   
18      /effects (2.73e-03)    spiele (2.12e-03)         .Take (2.70e-03)   
19        /legal (2.64e-03)       esl (2.09e-03)    /providers (2.67e-03)   

                                                                            \
                 ft_inv                      diff                 diff_inv   
0          .vn (0.0217)            Await (0.0428)              MR (0.0135)   
1    /Register (0.0106)           PLIC (6.54e-03)        strike (9.15e-03)   
2       sagt (6.51e-03)            =pd (6.42e-03)          Brew (8.07e-03)   
3     testim (6.13e-03)        eniable (5.45e-03)          .wik (6.70e-03)   
4      asign (4.92e-03)          Grain (4.19e-03)          fuse (5.27e-03)   
5        -ie (4.88e-03)           ickt (4.08e-03)        drinks (4.95e-03)   
6         że (4.32e-03)            hon (3.50e-03)        olicit (4.56e-03)   
7      zeigt (4.17e-03)           Pert (3.48e-03)             约 (4.11e-03)   
8       -ves (3.26e-03)          leine (3.01e-03)           Arr (4.10e-03)   
9          ť (3.09e-03)         orious (3.00e-03)       project (3.95e-03)   
10   personn (2.84e-03)  SystemService (2.76e-03)         drink (3.07e-03)   
11     probs (2.69e-03)             äß (2.68e-03)           ook (2.85e-03)   
12      elig (2.56e-03)          onces (2.64e-03)       Project (2.75e-03)   
13      roku (2.50e-03)          ernel (2.47e-03)          ford (2.56e-03)   
14     lesen (2.50e-03)           nite (2.33e-03)          boot (2.42e-03)   
15       )": (2.26e-03)         iliary (2.16e-03)         rouch (2.33e-03)   
16    ):\n\n (2.19e-03)        signage (2.12e-03)       Answers (2.23e-03)   
17       esl (2.14e-03)            --; (2.09e-03)   interesting (2.03e-03)   
18  ,,,,,,,, (2.11e-03)     Destructor (1.98e-03)          suit (1.88e-03)   
19    spiele (2.10e-03)          ücken (1.76e-03)        wooden (1.85e-03)   

             layer_14                                           \
                 base              base_inv                 ft   
0          , (0.8036)     contador (0.9623)         , (0.8160)   
1        ' ' (0.1076)    kontrol (3.11e-03)       ' ' (0.0975)   
2        the (0.0407)   karakter (2.50e-03)       the (0.0388)   
3        and (0.0302)       rekl (1.58e-03)       and (0.0302)   
4       in (6.04e-03)         bö (1.38e-03)      in (5.84e-03)   
5        ( (4.30e-03)       zoek (1.12e-03)       ( (5.44e-03)   
6       's (2.99e-03)     testim (9.58e-04)      's (1.76e-03)   
7        a (1.

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [6]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_7                                             \
                      base                         ft            diff   
0             The (0.1631)            .Today (0.0270)     -> (0.2819)   
1           There (0.0413)         Buccane (8.14e-03)     -> (0.0679)   
2            As (0.0409) ✅         .Second (7.26e-03)    1 (0.0333) ✅   
3          When (0.0214) ✅         /Area (4.84e-03) ✅   '\n' (0.0271)   
4            If (0.0172) ✅             aru (4.26e-03)     :- (0.0137)   
5              It (0.0170)             .au (4.01e-03)      . (0.0131)   
6         Given (0.0165) ✅            fter (3.61e-03)      ( (0.0119)   
7             For (0.0145)         /Math (2.87e-03) ✅    ' ' (0.0116)   
8              To (0.0141)    confidence (2.70e-03) ✅    : (9.58e-03)   
9            This (0.0131)      /problem (2.56e-03) ✅    : (9.38e-03)   
10             In (0.0123)     /problems (2.48e-03) ✅   (m (5.28e-03)   
11              A (0.0123)     /operator (2.46e-03) ✅  5 (4.64e-03) ✅   
12            You (0.0106)     /entities (2.24e-03) ✅  2 (3.68e-03) ✅   
13        While (0.0105) ✅         /bind (2.15e-03) ✅  4 (3.19e-03) ✅   
14       Having (0.0100) ✅     /activity (2.10e-03) ✅    > (3.14e-03)   
15   Although (9.70e-03) ✅            ilot (1.89e-03)   (g (3.05e-03)   
16          Let (9.01e-03)   persistence (1.78e-03) ✅  9 (3.00e-03) ✅   
17      Several (8.93e-03)            oire (1.72e-03)   :- (2.84e-03)   
18  According (8.06e-03) ✅      /respond (1.69e-03) ✅  6 (2.52e-03) ✅   
19            I (7.64e-03)             eft (1.67e-03)  3 (2.41e-03) ✅   

                layer_14                                                     \
                    base                    ft                         diff   
0          To (0.7292) ✅           To (0.5729)              NEWS (2.93e-03)   
1           ### (0.1217)           ** (0.2490)              frei (2.80e-03)   
2            ** (0.0708)          ### (0.1331)            acad (2.59e-03) ✅   
3         Let (0.0575) ✅        Let (0.0352) ✅          aktual (2.55e-03) ✅   
4           The (0.0145)        The (6.09e-03)            news (2.14e-03) ✅   
5   Certainly (1.35e-03)  Certainly (1.15e-03)             pos (2.12e-03) ✅   
6        Sure (1.05e-03)         ## (1.11e-03)        concrete (2.01e-03) ✅   
7          In (7.25e-04)       Sure (6.98e-04)         personn (1.88e-03) ✅   
8          ## (6.96e-04)    ---\n\n (3.90e-04)               rij (1.85e-03)   
9     Given (2.35e-04) ✅    First (1.32e-04) ✅            pron (1.85e-03) ✅   
10    First (2.21e-04) ✅       #### (1.22e-04)              News (1.76e-03)   
11          1 (1.26e-04)        ``` (1.22e-04)         barcelona (1.63e-03)   
12         We (1.26e-04)    Given (9.08e-05) ✅         Christoph (1.54e-03)   
13    Alright (1.16e-04)         In (9.08e-05)                /~ (1.43e-03)   
14       This (9.81e-05)        1 (8.71e-05) ✅               weg (1.42e-03)   
15       Here (9.81e-05)    Alright (6.24e-05)        Abstract (1.39e-03) ✅   
16       #### (9.01e-05)       Here (4.28e-05)               bre (1.37e-03)   
17        ``` (8.82e-05)         We (3.34e-05)         equival (1.31e-03) ✅   
18         As (6.72e-05)       This (2.29e-05)   Communication (1.25e-03) ✅   
19        For (6.32e-05)         As (2.29e-05)        Language (1.24e-03) ✅   

                layer_15                                                  
                    base                    ft                      diff  
0            To (0.4141)           ** (0.4023)                � (0.0947)  
1            ** (0.2852)          ### (0.2773)                ― (0.0610)  
2           ### (0.2500)           To (0.2773)       Language (0.0271) ✅  
3         Let (0.0265) ✅          The (0.0177)                2 (0.0211)  
4           The (0.0160)        Let (0.0177) ✅                1 (0.0204)  
5   Certainly (2.46e-03)  Certainly (1.65e-03)                ¹ (0.0170)  
6        Sure (1.49e-03)         In (9.99e-04)                — (0.0165) 

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [7]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {7: [0, 1, 2, 3, 4, 5], 14: [0, 1, 2, 3, 4, 5], 15: [0, 1, 2, 3, 4, 5]}


layer_7                             \
                           base                         ft   
0                  the (0.1154)                's (0.0473)   
1                 this (0.0337)        /problem (0.0340) ✅   
2                   's (0.0326)       /problems (0.0221) ✅   
3           /problem (0.0278) ✅       /entities (0.0200) ✅   
4              solve (0.0187) ✅           solve (0.0129) ✅   
5                :\n\n (0.0180)         /manage (0.0123) ✅   
6          /problems (0.0168) ✅               the (0.0105)   
7          /entities (0.0156) ✅             you (9.51e-03)   
8            /manage (0.0130) ✅              ’s (4.86e-03)   
9                you (8.66e-03)          .Today (4.59e-03)   
10                 , (8.15e-03)        solves (4.50e-03) ✅   
11             seems (5.99e-03)       /global (4.29e-03) ✅   
12      understand (5.53e-03) ✅  /preferences (4.21e-03) ✅   
13    /preferences (3.82e-03) ✅           seems (3.70e-03)   
14            .Today (3.72e-03)            your (3.69e-03)   
15         /global (3.71e-03) ✅               , (3.65e-03)   
16         analyze (3.68e-03) ✅     /provider (3.39e-03) ✅   
17              your (3.38e-03)       address (3.20e-03) ✅   
18          solves (3.36e-03) ✅    understand (3.07e-03) ✅   
19         address (3.01e-03) ✅       /crypto (2.68e-03) ✅   
20                ’s (2.88e-03)           :\n\n (2.66e-03)   
21         /crypto (2.88e-03) ✅       /object (2.37e-03) ✅   
22       /provider (2.35e-03) ✅   /connection (2.34e-03) ✅   
23               :\n (2.35e-03)      /effects (2.28e-03) ✅   
24            /job (2.28e-03) ✅          /job (2.24e-03) ✅   
25        /effects (2.17e-03) ✅  /application (1.72e-03) ✅   
26          tackle (2.01e-03) ✅      /logging (1.65e-03) ✅   
27     /connection (1.88e-03) ✅               : (1.63e-03)   
28           break (1.85e-03) ✅       /shared (1.53e-03) ✅   
29         /object (1.81e-03) ✅        /tasks (1.49e-03) ✅   
30         /layout (1.70e-03) ✅             /pr (1.47e-03)   
31        /logging (1.42e-03) ✅          /man (1.38e-03) ✅   
32                we (1.36e-03)       /engine (1.34e-03) ✅   
33          decode (1.16e-03) ✅              is (1.21e-03)   
34            begins (1.13e-03)       /layout (1.12e-03) ✅   
35         /shared (1.10e-03) ✅          begins (1.12e-03)   
36         /engine (1.10e-03) ✅            /con (1.11e-03)   
37                 : (1.07e-03)  /controllers (1.05e-03) ✅   
38    /controllers (1.05e-03) ✅     /activity (1.04e-03) ✅   
39       /activity (1.04e-03) ✅              we (1.03e-03)   
40         /entity (1.02e-03) ✅           /pl (1.01e-03) ✅   
41    /application (1.01e-03) ✅        /legal (1.01e-03) ✅   
42          /tasks (8.91e-04) ✅       analyze (1.00e-03) ✅   
43             /pl (8.75e-04) ✅             :\n (9.35e-04)   
44             looks (8.24e-04)        tackle (9.19e-04) ✅   
45           appears (7.85e-04)        solved (8.27e-04) ✅   
46    /environment (7.17e-04) ✅  /environment (6.82e-04) ✅   
47               /pr (6.41e-04)      /testing (6.22e-04) ✅   
48            /reg (5.91e-04) ✅          /reg (6.12e-04) ✅   
49        /respond (5.85e-04) ✅      WHATSOEVER (5.57e-04)   
50         problem (5.71e-04) ✅      /respond (5.30e-04) ✅   
51        WHATSOEVER (5.10e-04)     /customer (5.25e-04) ✅   
52       /customer (5.10e-04) ✅       /entity (5.09e-04) ✅   
53          /legal (5.10e-04) ✅      /vendors (4.76e-04) ✅   
54             these (4.60e-04)         /spec (4.54e-04) ✅   
55            .Round (4.55e-04)          .Round (4.46e-04)   
56           /spec (4.36e-04) ✅                              
57                 a (3.56e-04)                              
58            '\n\n' (3.17e-04)                              
59        problems (3.06e-04) ✅                              
60         solving (2.70e-04) ✅                              
61             !\n\n (1.55e-04)                              
62               our (1.38e-04)                         

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [8]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                               \
               pos_-3                 pos_-1                 pos_0   
0       indi (0.0227)        gons (8.85e-03)       lica (8.06e-03)   
1        cox (0.0166)     criptor (8.30e-03)       cate (8.06e-03)   
2        gib (0.0121)      etting (4.43e-03)       xima (7.11e-03)   
3    inite (8.85e-03)      gorith (4.18e-03)        sel (7.11e-03)   
4      icc (7.35e-03)         ija (3.68e-03)       Pert (7.11e-03)   
5     init (7.35e-03)    presenta (3.68e-03)       hazi (6.68e-03)   
6      ERM (6.10e-03)         sel (3.05e-03)      urses (6.26e-03)   
7    onnen (5.07e-03)   continuar (3.05e-03)       gons (5.89e-03)   
8     mess (4.76e-03)          MS (2.87e-03)       dana (3.57e-03)   
9     avar (4.46e-03)         ={` (2.87e-03)        ixa (3.57e-03)   
10    raph (4.18e-03)      solete (2.87e-03)        GRE (3.57e-03)   
11      ’n (3.94e-03)   chedulers (2.87e-03)       imum (3.36e-03)   
12    scop (3.48e-03)      ogonal (2.87e-03)   enerated (3.36e-03)   
13    ween (3.27e-03)         won (2.69e-03)        agt (2.78e-03)   
14    indo (2.70e-03)        kich (2.53e-03)      enate (2.78e-03)   
15   Lilly (2.55e-03)        över (2.53e-03)   Mediterr (2.78e-03)   
16     ree (2.55e-03)         pak (2.38e-03)      Await (2.78e-03)   
17      ền (2.38e-03)       anoia (2.09e-03)        raq (2.61e-03)   
18    inee (2.38e-03)        Plus (2.09e-03)     iliary (2.61e-03)   
19     ESH (2.24e-03)         HAV (1.97e-03)    criptor (2.46e-03)   

                                                                        \
                   pos_1                 pos_2                   pos_3   
0          Pert (0.0126)         Pert (0.0181)           Pert (0.0142)   
1           ixa (0.0104)        Await (0.0170)          Await (0.0117)   
2        cate (8.12e-03)        --; (5.89e-03)        leine (8.06e-03)   
3         GRE (8.12e-03)        urf (5.52e-03)          --; (7.57e-03)   
4         urf (5.95e-03)      leine (4.30e-03)          urf (5.22e-03)   
5      sovere (5.58e-03)      arine (4.03e-03)      signage (4.88e-03)   
6       leine (5.25e-03)        ixa (4.03e-03)         uard (4.88e-03)   
7    enerated (4.36e-03)        GRE (3.36e-03)   Destructor (4.61e-03)   
8       Await (4.36e-03)     sovere (2.96e-03)        ücken (4.61e-03)   
9       urses (4.09e-03)      .urls (2.78e-03)        itere (4.30e-03)   
10      itere (4.09e-03)      annes (2.61e-03)          GRE (4.30e-03)   
11     urette (3.39e-03)   Typeface (2.61e-03)           hv (3.81e-03)   
12       Indo (3.19e-03)        .\" (2.46e-03)          ixa (3.81e-03)   
13   italiani (2.99e-03)       cate (2.46e-03)      Verdana (3.36e-03)   
14    signage (2.47e-03)    .')\n\n (2.46e-03)        Modes (3.36e-03)   
15        --; (2.33e-03)         hv (2.17e-03)        arine (3.36e-03)   
16      abide (2.33e-03)    signage (2.17e-03)      eniable (2.62e-03)   
17       xima (2.33e-03)        ccp (2.03e-03)         gons (2.46e-03)   
18      arine (2.33e-03)   italiani (1.91e-03)     Typeface (2.46e-03)   
19       uras (2.33e-03)      Modes (1.79e-03)           tu (2.30e-03)   

                                                      \
                    pos_10                    pos_50   
0           Await (0.0618)            Await (0.0479)   
1            Pert (0.0101)            =pd (8.85e-03)   
2         leine (7.87e-03)           PLIC (7.81e-03)   
3         itere (6.93e-03)        eniable (5.71e-03)   
4       signage (5.74e-03)            hon (4.18e-03)   
5           --; (5.40e-03)           ickt (3.94e-03)   
6           ixa (4.49e-03)          Grain (3.94e-03)   
7         ücken (4.49e-03)             äß (3.46e-03)   
8       eniable (3.94e-03)          leine (2.87e-03)   
9    Destructor (3.27e-03)         orious (2.70e-03)   
10          urf (3.27e-03)  SystemService (2.70e-03)   
11       ichtet (3.07e-03)          ernel (2.53e-03)   
12         atum (2.88e-03)          onces (2.38e-03)   
13  

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [9]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {PS_DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3]


layer_7                                        \
                    pos_-3          pos_-1                 pos_0   
0               , (0.0518)     -> (0.2819)          _ch (0.0316)   
1       subject (0.0107) ✅     -> (0.0679)            C (0.0244)   
2            sn (5.02e-03)    1 (0.0333) ✅       Chat (0.0183) ✅   
3            lo (4.32e-03)   '\n' (0.0271)           _c (0.0158)   
4     Caribbean (4.12e-03)     :- (0.0137)          .ch (0.0114)   
5     answers (4.12e-03) ✅      . (0.0131)    talking (0.0102) ✅   
6             b (3.29e-03)      ( (0.0119)          , (7.98e-03)   
7          Loop (3.23e-03)    ' ' (0.0116)          B (7.85e-03)   
8           sir (3.04e-03)    : (9.58e-03)          L (7.58e-03)   
9             , (3.02e-03)    : (9.38e-03)    chatt (7.05e-03) ✅   
10   question (2.97e-03) ✅   (m (5.28e-03)      Solar (5.09e-03)   
11       post (2.95e-03) ✅  5 (4.64e-03) ✅       bond (5.00e-03)   
12          loo (2.93e-03)  2 (3.68e-03) ✅          G (4.48e-03)   
13           HL (2.90e-03)  4 (3.19e-03) ✅       Bond (4.28e-03)   
14         mere (2.74e-03)    > (3.14e-03)    chats (3.84e-03) ✅   
15         loaf (2.66e-03)   (g (3.05e-03)  Talking (3.79e-03) ✅   
16            � (2.65e-03)  9 (3.00e-03) ✅       _com (3.75e-03)   
17            L (2.59e-03)   :- (2.84e-03)        _sh (3.60e-03)   
18            < (2.54e-03)  6 (2.52e-03) ✅          T (3.59e-03)   
19   substances (2.41e-03)  3 (2.41e-03) ✅         _b (3.34e-03)   

                                                                   \
                  pos_1                pos_2                pos_3   
0           -> (0.2485)          -> (0.5199)         ann (0.1127)   
1       blue (0.1823) ✅      blue (0.2188) ✅           " (0.1112)   
2        hello (0.1016)       hello (0.0693)          -> (0.0285)   
3        hello (0.0399)         ann (0.0270)           ' (0.0131)   
4            " (0.0379)          -> (0.0183)          _l (0.0129)   
5            ' (0.0177)           " (0.0111)   green (9.76e-03) ✅   
6           -> (0.0140)         ( (7.96e-03)       _re (8.18e-03)   
7      green (0.0117) ✅         , (7.91e-03)     hello (7.93e-03)   
8          : (4.07e-03)         ' (6.37e-03)         _ (7.54e-03)   
9          = (3.65e-03)       ' ' (5.09e-03)         , (6.60e-03)   
10       ' ' (3.56e-03)   green (3.77e-03) ✅        _n (6.08e-03)   
11         ( (3.33e-03)         . (2.23e-03)       ' ' (6.06e-03)   
12        => (3.29e-03)         : (2.06e-03)        _f (5.55e-03)   
13     red (2.96e-03) ✅     red (1.48e-03) ✅        _m (4.12e-03)   
14         ` (2.69e-03)   goodbye (1.35e-03)       _an (3.94e-03)   
15        (" (2.00e-03)        is (1.33e-03)        _v (3.42e-03)   
16         - (1.82e-03)         - (1.28e-03)        _s (3.37e-03)   
17         > (1.78e-03)         = (1.09e-03)    blue (3.36e-03) ✅   
18   goodbye (1.65e-03)         > (1.02e-03)        _h (2.95e-03)   
19       man (1.65e-03)         : (1.00e-03)    _red (2.82e-03) ✅   

                      layer_14                               \
                        pos_-3                       pos_-1   
0              fatt (8.50e-03)              NEWS (2.93e-03)   
1               cre (6.39e-03)              frei (2.80e-03)   
2          universe (5.98e-03)            acad (2.59e-03) ✅   
3                 b (5.41e-03)          aktual (2.55e-03) ✅   
4              doen (5.27e-03)            news (2.14e-03) ✅   
5        literacy (4.55e-03) ✅             pos (2.12e-03) ✅   
6          membrane (4.26e-03)        concrete (2.01e-03) ✅   
7           spell (4.15e-03) ✅         personn (1.88e-03) ✅   
8               177 (3.96e-03)               rij (1.85e-03)   
9           chatter (3.64e-03)            pron (1.85e-03) ✅   
10       spelling (3.10e-03) ✅              News (1.76e-03)   
11   dictionaries (3.09e-03) ✅         barcelona (1.63e-03)   
12          remin (3.01e-03) ✅         Christoph (1.54e-03)   
13                B (2.98e-03)                /~ (1.43e-03